# Example for Prediction of SARS-CoV-2 Epitope-Specific TCR Recognition Using a Pre-Trained Protein Language Model

- **Reference**: Yoo, S., Jeong, M., Seomun, S., Kim, K., & Han, Y. (2024). Interpretable Prediction of SARS-CoV-2 Epitope-specific TCR Recognition Using a Pre-Trained Protein Language Model. IEEE/ACM Transactions on Computational Biology and Bioinformatics.

## Configuration Setup

In [5]:
!pip install -r requirements.txt

In [ ]:
# import libraries
import random
import os
import torch

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    roc_auc_score,
    average_precision_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EsmForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    get_scheduler
)

from peft import get_peft_model, LoraConfig, PeftModel
from datasets import Dataset, DatasetDict
from evaluate import load

/home/hym/miniconda3/envs/TCRBert/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load sample fine-tuning data

In [2]:
from IPython.display import display

df = pd.read_csv('train.sample.csv')
print(len(df))
display(df.head())

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

print(len(train_df))
print(len(val_df))

# convert data into HuggingFace DatasetDict
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
ds_dict = DatasetDict({
    'train': train_ds,
    'val': val_ds
})
display(ds_dict)

252


,epitope,epitope_gene,epitope_species,species,cdr3b,mhc,source,label
0,LPRRSGAAGA,NaN,Influenza,Human,CASSEGLLYEQYF,HLA-B7,McPAS,1
1,SFHSLHLLF,NaN,HTLV-1,Human,CASGLGTDTQYF,HLA-A*24:02,McPAS,1
2,GILGFVFTL,NaN,Influenza,Human,CASGGTGIDEAFF,HLA-A*02:05,Control,0
3,GILGFVFTL,M,InfluenzaA,HomoSapiens,CASSIRSGETQYF,HLA-A*02:01,VDJdb,1
4,GLCTLVAML,NaN,Epstein Barr virus (EBV),Human,CASSLSVGGEQFF,HLA-A2,Control,0


201
51


DatasetDict({
    train: Dataset({
        features: ['epitope', 'epitope_gene', 'epitope_species', 'species', 'cdr3b', 'mhc', 'source', 'label', '__index_level_0__'],
        num_rows: 201
    })
    val: Dataset({
        features: ['epitope', 'epitope_gene', 'epitope_species', 'species', 'cdr3b', 'mhc', 'source', 'label', '__index_level_0__'],
        num_rows: 51
    })
})

## Load tokenizer and tokenize data

In [3]:
import os

model_id = "facebook/esm2_t6_8M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(examples, max_length=80):
    text = [e + c for e, c in zip(examples["epitope"], examples["cdr3b"])]
    encoding = tokenizer(text, truncation=True, max_length=max_length)
    encoding["labels"] = examples["label"]
    return encoding

encoded_ds = ds_dict.map(
    tokenize,
    batched=True,
    num_proc=os.cpu_count(),
    remove_columns=ds_dict["train"].column_names
)

encoded_ds

Map (num_proc=24): 100%|█████████████████████████████████████████████| 51/51 [00:00<00:00, 126.27 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 201
    })
    val: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 51
    })
})

## Fine-tuning the pre-trained model

### Load pre-trained model checkpoint and configure the model for LoRA fine-tuning

In [4]:
model = EsmForSequenceClassification.from_pretrained(model_id, num_labels=2)

peft_config = LoraConfig(
    task_type="SEQ_CLS",
    inference_mode=False,
    bias="none",
    r=8, # rank number
    lora_alpha=16, # scaling factor
    lora_dropout=0.2, # dropout prob
    target_modules=[ # which layers to apply LoRA
        "query",
        "key",
        "value"
    ],
    modules_to_save=['classifier'] # ensures that the fine-tuned classifier head is saved when calling trainer.save_model later
)

model = get_peft_model(model, peft_config)

# adjust dropout in the classifier head
model.base_model.model.classifier.modules_to_save.default.dropout.p = 0.25

# show amount of trainable parameters
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

print_trainable_parameters(model)

Loading weights: 100%|██████████████████████████████████████████████████| 107/107 [00:00<00:00, 14529.61it/s]
EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 195522 || all params: 7707965 || trainable%: 2.536622831058522


### LoRA fine-tuning the pre-trained model 

In [5]:
# Define metrics to compute during training
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    softmax = torch.nn.Softmax(dim=1)
    probabilities = softmax(torch.tensor(logits.copy())).numpy()
    predictions = np.argmax(probabilities, axis=1)
    probabilities_pos_class = probabilities[:, 1]

    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, zero_division=0)
    recall = recall_score(labels, predictions, zero_division=0)
    auc = roc_auc_score(labels, probabilities_pos_class)
    auc_pr = average_precision_score(labels, probabilities_pos_class)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "auc_roc": auc,
        "auc_pr": auc_pr
    }


num_train_epochs = 10
batch_size = 16
learning_rate = 1e-3

# Early stopping
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=max(1, int(num_train_epochs*0.2)),
    early_stopping_threshold=0.01
)

# Training args
args = TrainingArguments(
    seed=42,
    fp16=True,
    output_dir='./results',
    eval_strategy="steps",
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=4,
    # gradient_checkpointing=True,
    logging_steps=50,
    eval_steps=50,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    metric_for_best_model="auc_roc",
    load_best_model_at_end=True,
    report_to='none'  # Disable Weights & Biases logging
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create trainer and fine-tune the model
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_ds['train'],
    eval_dataset=encoded_ds['val'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping_callback]
)

# train model
trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/home/hym/miniconda3/envs/TCRBert/lib/python3.11/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2}.
/home/hym/miniconda3/envs/TCRBert/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarn

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,Auc Roc,Auc Pr
20,No log,0.701409,0.509804,0.509804,1.000000,0.507692,0.516256


TrainOutput(global_step=20, training_loss=0.6875614643096923, metrics={'train_runtime': 8.8837, 'train_samples_per_second': 226.257, 'train_steps_per_second': 2.251, 'total_flos': 2792633928810.0, 'train_loss': 0.6875614643096923, 'epoch': 10.0})